# Hotel Booking Demand — Exploratory Data Analysis (EDA)

**Project:** Python & Pandas EDA  
**Dataset source:** [Hotel Booking Demand — Kaggle](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Objective:** Explore booking patterns, missing data, pricing, and cancellation behaviour using descriptive analysis and visualisations.

> **Important:** This is exploratory analysis, not proof of causation. Relationships shown in charts are associations in this historical dataset. The dataset includes bookings for a city hotel and a resort hotel in Portugal, covering arrivals from 2015–2017.


## 1. Business questions

1. How many bookings and variables are in the dataset, and what are their data types?
2. Which columns contain missing values, and how severe is the missingness?
3. Do cancellation rates differ between City and Resort hotels?
4. Is booking lead time associated with cancellation?
5. How do cancellation rates vary by deposit type and market segment?
6. Which arrival months have the highest booking volume?
7. How are nightly rates distributed, and how do special requests relate to cancellations?
8. Which numeric variables show the strongest pairwise correlations?

## 2. Setup and load the dataset

Download `hotel_bookings.csv` from the Kaggle dataset page linked above. In Google Colab, upload the CSV when prompted; in Jupyter, place it in the same folder as this notebook.


In [ ]:
# If needed, install these packages in a notebook cell:
# %pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("hotel_bookings.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Google Colab upload fallback
    try:
        from google.colab import files
        print("Upload the hotel_bookings.csv file you downloaded from Kaggle.")
        uploaded = files.upload()
        csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
        if csv_name is None:
            raise FileNotFoundError("No CSV file was uploaded. Upload hotel_bookings.csv.")
        df = pd.read_csv(csv_name)
    except ImportError:
        raise FileNotFoundError(
            "hotel_bookings.csv was not found. Download it from Kaggle and place it "
            "in the same folder as this notebook, or update DATA_PATH."
        )

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}")
df.head()


## 3. Structural overview

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn data types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nDataFrame information:")
df.info()

print("\nMemory usage:")
memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Approximate memory usage: {memory_mb:.2f} MB")

print("\nFirst five rows:")
display(df.head())

print("\nLast five rows:")
display(df.tail())

print("\nNumeric summary statistics:")
display(df.describe().T)

print("\nCategorical summary:")
display(df.describe(include=["object"]).T)


## 4. Data quality and missing values

The raw dataset has columns such as `company` and `agent` where missing values can mean that a booking was not associated with a recorded company or agent. We will report missingness first rather than automatically filling every missing value. Four missing values are commonly reported for `children`; this notebook checks the actual file loaded.


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])

plt.figure(figsize=(11, 6))
missing_plot = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_percent")
if len(missing_plot):
    plt.barh(missing_plot.index, missing_plot["missing_percent"])
    plt.xlabel("Missing values (%)")
    plt.ylabel("Column")
    plt.title("Missing-value percentage by column")
    plt.tight_layout()
    plt.show()
    top_missing = missing_summary[missing_summary["missing_count"] > 0].iloc[0]
    print(
        f"Interpretation: '{missing_summary.index[0]}' has the largest missing-value "
        f"share among the columns listed ({top_missing['missing_percent']:.2f}%). "
        "Missingness is concentrated in a small number of fields, so handling should be column-specific."
    )
else:
    print("Interpretation: No missing values were detected in the loaded dataset.")


## 5. Prepare analysis fields

In [ ]:
# Work on a copy and preserve the original dataframe.
eda = df.copy()

# Convert the binary cancellation indicator to a numeric value if needed.
if "is_canceled" in eda.columns:
    eda["is_canceled"] = pd.to_numeric(eda["is_canceled"], errors="coerce")

# Convert children missing values to 0 only in a derived analysis copy.
# This is a documented assumption: missing children counts are treated as no children
# for analyses that require guest counts, not as a universal replacement for all fields.
if "children" in eda.columns:
    eda["children"] = eda["children"].fillna(0)

# Build total stay duration where the required columns exist.
stay_cols = ["stays_in_weekend_nights", "stays_in_week_nights"]
if all(c in eda.columns for c in stay_cols):
    eda["total_nights"] = eda[stay_cols].sum(axis=1)

# Remove impossible negative ADR values only from the ADR-specific visualisation later;
# preserve raw rows in the main dataframe and report how many are present.
if "adr" in eda.columns:
    eda["adr"] = pd.to_numeric(eda["adr"], errors="coerce")

print("Rows in analysis copy:", len(eda))
print("Exact duplicate rows:", int(eda.duplicated().sum()))
if "adr" in eda.columns:
    print("Rows with negative ADR:", int((eda["adr"] < 0).sum()))


## 6. Visual analysis

Each chart is designed to answer a specific question. The interpretation is printed directly below each visual and is calculated from the loaded dataset.


### Visualisation 1 — What proportion of bookings were cancelled by hotel type?

In [ ]:
if {"hotel", "is_canceled"}.issubset(eda.columns):
    cancel_by_hotel = eda.groupby("hotel", dropna=False)["is_canceled"].mean().mul(100).sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    plt.bar(cancel_by_hotel.index.astype(str), cancel_by_hotel.values)
    plt.ylabel("Cancelled bookings (%)")
    plt.xlabel("Hotel type")
    plt.title("Cancellation rate by hotel type")
    plt.ylim(0, max(5, cancel_by_hotel.max() * 1.15))
    plt.tight_layout()
    plt.show()
    if len(cancel_by_hotel) >= 2:
        gap = cancel_by_hotel.iloc[0] - cancel_by_hotel.iloc[-1]
        print(f"Interpretation: {cancel_by_hotel.index[0]} has the higher observed cancellation rate "
              f"({cancel_by_hotel.iloc[0]:.1f}%), a difference of {gap:.1f} percentage points "
              f"from {cancel_by_hotel.index[-1]}. This is a descriptive difference, not evidence of causation.")
else:
    print("Required columns are not available for this chart.")


### Visualisation 2 — Is longer booking lead time associated with cancellation?

In [ ]:
if {"lead_time", "is_canceled"}.issubset(eda.columns):
    plt.figure(figsize=(9, 5))
    for status, label in [(0, "Not cancelled"), (1, "Cancelled")]:
        vals = eda.loc[eda["is_canceled"] == status, "lead_time"].dropna()
        vals = vals[vals >= 0]
        if len(vals):
            plt.hist(vals, bins=40, alpha=0.55, density=True, label=label)
    plt.xlabel("Lead time (days)")
    plt.ylabel("Density")
    plt.title("Booking lead-time distribution by cancellation status")
    plt.legend()
    plt.tight_layout()
    plt.show()
    medians = eda.groupby("is_canceled")["lead_time"].median()
    if 0 in medians.index and 1 in medians.index:
        print(f"Interpretation: Median lead time is {medians.loc[1]:.0f} days for cancelled bookings "
              f"and {medians.loc[0]:.0f} days for bookings that were not cancelled. "
              "Compare the distributions as an association; lead time alone does not explain every cancellation.")
else:
    print("Required columns are not available for this chart.")


### Visualisation 3 — How does deposit type relate to cancellation rate?

In [ ]:
if {"deposit_type", "is_canceled"}.issubset(eda.columns):
    deposit_cancel = (eda.groupby("deposit_type", dropna=False)["is_canceled"]
                      .mean().mul(100).sort_values(ascending=False))
    plt.figure(figsize=(9, 5))
    plt.bar(deposit_cancel.index.astype(str), deposit_cancel.values)
    plt.xlabel("Deposit type")
    plt.ylabel("Cancelled bookings (%)")
    plt.title("Cancellation rate by deposit type")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
    print(f"Interpretation: The highest observed cancellation rate is for '{deposit_cancel.index[0]}' "
          f"({deposit_cancel.iloc[0]:.1f}%). Deposit policies may be related to booking outcomes, "
          "but customer mix and other factors may also contribute.")
else:
    print("Required columns are not available for this chart.")


### Visualisation 4 — Which arrival months have the most bookings?

In [ ]:
if {"arrival_date_month"}.issubset(eda.columns):
    month_order = ["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"]
    month_counts = eda["arrival_date_month"].value_counts()
    ordered_months = [m for m in month_order if m in month_counts.index]
    # Append unexpected labels at the end so no category is silently omitted.
    ordered_months += [m for m in month_counts.index if m not in ordered_months]
    month_counts = month_counts.reindex(ordered_months)
    plt.figure(figsize=(11, 5))
    plt.bar(month_counts.index.astype(str), month_counts.values)
    plt.xlabel("Arrival month")
    plt.ylabel("Number of bookings")
    plt.title("Booking volume by arrival month")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
    print(f"Interpretation: '{month_counts.idxmax()}' has the highest booking count "
          f"({month_counts.max():,}) in the loaded data. This reflects booking records in the dataset "
          "period, not necessarily a general pattern for every year or destination.")
else:
    print("Required column is not available for this chart.")


### Visualisation 5 — What does the average daily rate (ADR) distribution look like?

In [ ]:
if "adr" in eda.columns:
    adr_valid = eda["adr"].dropna()
    adr_valid = adr_valid[adr_valid >= 0]
    plt.figure(figsize=(9, 5))
    plt.hist(adr_valid, bins=50)
    plt.xlabel("Average daily rate (ADR)")
    plt.ylabel("Number of bookings")
    plt.title("Distribution of average daily rate (non-negative values)")
    plt.tight_layout()
    plt.show()
    if len(adr_valid):
        median_adr = adr_valid.median()
        mean_adr = adr_valid.mean()
        print(f"Interpretation: Mean ADR is {mean_adr:.2f} and median ADR is {median_adr:.2f}. "
              + ("The mean exceeds the median, suggesting a right-skewed distribution with higher-rate bookings."
                 if mean_adr > median_adr else
                 "The mean does not exceed the median, so the distribution is not strongly right-skewed by this simple comparison."))
else:
    print("ADR column is not available.")


### Visualisation 6 — How do cancellation rates vary by market segment?

In [ ]:
if {"market_segment", "is_canceled"}.issubset(eda.columns):
    segment_summary = (eda.groupby("market_segment", dropna=False)
                       .agg(bookings=("is_canceled", "size"),
                            cancellation_rate=("is_canceled", "mean"))
                       .assign(cancellation_rate=lambda x: x["cancellation_rate"] * 100)
                       .sort_values("cancellation_rate", ascending=False))
    plt.figure(figsize=(10, 5))
    plt.bar(segment_summary.index.astype(str), segment_summary["cancellation_rate"])
    plt.xlabel("Market segment")
    plt.ylabel("Cancelled bookings (%)")
    plt.title("Cancellation rate by market segment")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    top_seg = segment_summary.index[0]
    print(f"Interpretation: '{top_seg}' has the highest observed cancellation rate "
          f"({segment_summary.iloc[0]['cancellation_rate']:.1f}%) among the displayed market segments. "
          "Check the booking count for each segment before drawing strong conclusions.")
    display(segment_summary)
else:
    print("Required columns are not available for this chart.")


### Visualisation 7 — Are special requests associated with cancellation?

In [ ]:
if {"total_of_special_requests", "is_canceled"}.issubset(eda.columns):
    request_summary = (eda.groupby("total_of_special_requests", dropna=False)["is_canceled"]
                       .agg(["count", "mean"]).rename(columns={"count": "bookings", "mean": "cancellation_rate"}))
    request_summary["cancellation_rate"] *= 100
    plt.figure(figsize=(9, 5))
    plt.bar(request_summary.index.astype(str), request_summary["cancellation_rate"])
    plt.xlabel("Number of special requests")
    plt.ylabel("Cancelled bookings (%)")
    plt.title("Cancellation rate by number of special requests")
    plt.tight_layout()
    plt.show()
    valid_summary = request_summary.dropna(subset=["cancellation_rate"])
    if len(valid_summary) >= 2:
        print(f"Interpretation: The cancellation rate is {valid_summary.iloc[0]['cancellation_rate']:.1f}% "
              f"at {valid_summary.index[0]} special requests and {valid_summary.iloc[-1]['cancellation_rate']:.1f}% "
              f"at {valid_summary.index[-1]} requests. These groups may differ in other ways, so this is not a causal claim.")
    display(request_summary)
else:
    print("Required columns are not available for this chart.")


### Visualisation 8 — Which numeric features are correlated?

In [ ]:
numeric = eda.select_dtypes(include=np.number)
# Limit to a readable set of commonly useful numeric columns when available.
preferred = ["is_canceled", "lead_time", "adr", "total_nights", "adults", "children",
             "babies", "previous_cancellations", "booking_changes",
             "required_car_parking_spaces", "total_of_special_requests", "days_in_waiting_list"]
corr_cols = [c for c in preferred if c in numeric.columns]
corr_data = numeric[corr_cols] if len(corr_cols) >= 2 else numeric
corr = corr_data.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", square=False)
plt.title("Correlation heatmap for selected numeric variables")
plt.tight_layout()
plt.show()

if "is_canceled" in corr.columns:
    related = corr["is_canceled"].drop("is_canceled").dropna()
    if len(related):
        strongest = related.abs().sort_values(ascending=False).index[0]
        print(f"Interpretation: Among the numeric variables shown, '{strongest}' has the largest "
              f"absolute correlation with cancellation ({corr.loc[strongest, 'is_canceled']:.2f}). "
              "Correlation measures linear association and does not establish cause and effect.")
else:
    print("Interpretation: Review the heatmap for positive and negative relationships; cancellation is not available as a numeric column.")


## 7. Categorical value counts and additional summary

This section explicitly reviews key categorical fields, as required, and includes robust handling if a field is not present.


In [ ]:
for col in ["hotel", "meal", "market_segment", "distribution_channel",
             "deposit_type", "customer_type", "reserved_room_type", "reservation_status"]:
    if col in eda.columns:
        print(f"\nValue counts for {col}:")
        display(eda[col].value_counts(dropna=False).head(15).rename("count").to_frame())

print("\nNumeric summary (mean, median, standard deviation, quartiles):")
num_summary = eda.select_dtypes(include=np.number).describe().T
wanted_stats = [c for c in ["count", "mean", "50%", "std", "25%", "75%", "min", "max"] if c in num_summary.columns]
display(num_summary[wanted_stats].rename(columns={"50%": "median", "25%": "Q1", "75%": "Q3"}))


## 8. Key findings — generated from this dataset

The findings below are computed from the loaded CSV, so they reflect the actual file you analyse. Review the outputs and add your own short explanation before submitting.


In [ ]:
findings = []

if {"hotel", "is_canceled"}.issubset(eda.columns):
    rates = eda.groupby("hotel")["is_canceled"].mean() * 100
    if len(rates):
        high_hotel = rates.idxmax()
        findings.append(f"Cancellation rates differ by hotel type; {high_hotel} is highest at {rates.max():.1f}%.")

if {"lead_time", "is_canceled"}.issubset(eda.columns):
    med = eda.groupby("is_canceled")["lead_time"].median()
    if 0 in med.index and 1 in med.index:
        findings.append(f"Median lead time is {med.loc[1]:.0f} days for cancelled bookings versus {med.loc[0]:.0f} days for non-cancelled bookings.")

if "arrival_date_month" in eda.columns and len(eda):
    counts = eda["arrival_date_month"].value_counts()
    findings.append(f"{counts.idxmax()} has the highest number of booking records ({counts.max():,}).")

if "deposit_type" in eda.columns and "is_canceled" in eda.columns:
    rates = eda.groupby("deposit_type")["is_canceled"].mean() * 100
    if len(rates):
        findings.append(f"The highest cancellation rate by deposit type is {rates.idxmax()} ({rates.max():.1f}%).")

if "adr" in eda.columns:
    adr = eda["adr"].dropna()
    adr = adr[adr >= 0]
    if len(adr):
        findings.append(f"Non-negative ADR has a mean of {adr.mean():.2f} and a median of {adr.median():.2f}.")

if "total_of_special_requests" in eda.columns and "is_canceled" in eda.columns:
    rates = eda.groupby("total_of_special_requests")["is_canceled"].mean() * 100
    if len(rates) >= 2:
        findings.append(f"Cancellation rate varies across special-request counts, from {rates.min():.1f}% to {rates.max():.1f}% across observed groups.")

print("KEY FINDINGS")
for i, finding in enumerate(findings[:6], start=1):
    print(f"{i}. {finding}")

print("\nFinal interpretation notes:")
print("- These are descriptive findings from historical bookings, not causal conclusions.")
print("- Missing values and exact duplicates were reported separately; any data cleaning decision should be documented.")
print("- The dataset covers a specific time period and two Portuguese hotels, so results may not generalise to other markets.")


## 9. Conclusion

This EDA examined the structure and quality of hotel booking data, summarised numeric and categorical variables, and visualised booking volume, cancellation behaviour, lead time, deposit type, market segment, ADR, special requests, and numeric correlations. The main value of EDA is to identify patterns and data-quality issues that deserve further investigation.

### Before submitting
- [ ] Run every cell from top to bottom without errors.
- [ ] Check that the dataset has at least 1,000 rows and 8 columns.
- [ ] Ensure all charts have a title and labelled axes.
- [ ] Read the printed interpretations and rewrite/add your own observations in your own words.
- [ ] Add a screenshot of one or two important charts to the README if desired.
- [ ] Push the notebook and README to a public GitHub repository.
